<h1>Introduction to dplyr</h1>

<small>Source: <a href="https://github.com/tidyverse/dplyr/blob/pkgdown-v1.1.4/vignettes/dplyr.Rmd"><code>vignettes/dplyr.Rmd</code></a></small>
<div><code>dplyr.Rmd</code></div>


<p>When working with data you must:</p>

<ul>
<li><p>Figure out what you want to do.</p></li>
<li><p>Describe those tasks in the form of a computer program.</p></li>
<li><p>Execute the program.</p></li>
</ul>

<p>The dplyr package makes these steps fast and easy:</p>

<ul>
<li><p>By constraining your options, it helps you think about your data manipulation challenges.</p></li>
<li><p>It provides simple “verbs”, functions that correspond to the most common data manipulation tasks, to help you translate your thoughts into
code.</p></li>
<li><p>It uses efficient backends, so you spend less time waiting for the computer.</p></li>
</ul>

<p>This document introduces you to dplyr’s basic set of tools, and shows you how to apply them to data frames. dplyr also supports databases via
the dbplyr package, once you’ve installed, read <code>vignette("dbplyr")</code> to learn more.</p>

# <h2 id="data-starwars">Data: starwars</h2>

<p>To explore the basic data manipulation verbs of dplyr, we’ll use the dataset <code>starwars</code>. This dataset contains 87 characters and
comes from the <a href="https://swapi.dev">Star Wars API</a>, and is documented in <code><a href="https://dplyr.tidyverse.org/reference/starwars.html">?starwars</a></code></p>

In [34]:
library(dplyr)

In [35]:
dim(starwars)

[1] 87 14

In [3]:
starwars

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Luke Skywalker,172,77.0,blond,fair,blue,19.0,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens","Snowspeeder , Imperial Speeder Bike","X-wing , Imperial shuttle"
C-3PO,167,75.0,NA,gold,yellow,112.0,none,masculine,Tatooine,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",,
R2-D2,96,32.0,NA,"white, blue",red,33.0,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,
Darth Vader,202,136.0,none,white,yellow,41.9,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope",,TIE Advanced x1
Leia Organa,150,49.0,brown,light,brown,19.0,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,
Owen Lars,178,120.0,"brown, grey",light,blue,52.0,male,masculine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
Beru Whitesun lars,165,75.0,brown,light,blue,47.0,female,feminine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
R5-D4,97,32.0,NA,"white, red",red,NA,none,masculine,Tatooine,Droid,A New Hope,,
Biggs Darklighter,183,84.0,black,light,brown,24.0,male,masculine,Tatooine,Human,A New Hope,,X-wing


<p>Note that <code>starwars</code> is a tibble, a modern reimagining of the data frame. It’s particularly useful for large datasets because it
only prints the first few rows. You can learn more about tibbles at <a href="https://tibble.tidyverse.org">https://tibble.tidyverse.org</a>; in particular you can convert data frames to tibbles with <code><a href="https://tibble.tidyverse.org/reference/as_tibble.html">as_tibble()</a></code>.</p>


# <h2 id="single-table-verbs">Single table verbs</h2>

<p>dplyr aims to provide a function for each basic verb of data manipulation. These verbs can be organised into three categories based on the component of the dataset that they work with:</p>

<ul>
    
<li>Rows:
<ul>
<li><code><a href="https://dplyr.tidyverse.org/reference/filter.html">filter()</a></code> chooses rows based on column values.</li>
<li><code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice()</a></code> chooses rows based on location.</li>
<li><code><a href="https://dplyr.tidyverse.org/reference/arrange.html">arrange()</a></code> changes the order of the rows.</li>
</ul>
</li>

<li>Columns:
<ul>
<li><code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> changes whether or not a column is included.</li>
<li><code><a href="https://dplyr.tidyverse.org/reference/rename.html">rename()</a></code> changes the name of columns.</li>
<li><code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code> changes the values of columns and creates new columns.</li>
<li><code><a href="https://dplyr.tidyverse.org/reference/relocate.html">relocate()</a></code> changes the order of the columns.</li>
</ul>
</li>
                 
<li>Groups of rows:
<ul>
<li><code><a href="https://dplyr.tidyverse.org/reference/summarise.html">summarise()</a></code> collapses a group into a single row.</li>
</ul>
</li>
                 
</ul>


## <h3 id="the-pipe">The pipe</h3>

<p>All of the dplyr functions take a data frame (or tibble) as the first argument. Rather than forcing the user to either save intermediate
objects or nest functions, dplyr provides the <code>%&gt;%</code> operator from magrittr. <code>x %&gt;% f(y)</code> turns into <code>f(x, y)</code> so the result from one step is then “piped” into the next step. You can use the pipe to rewrite multiple operations that you can read left-to-right, top-to-bottom (reading the pipe operator as “then”).</p>


## <h3 id="filter-rows-with-filter">Filter rows with <code>filter()</code></h3>

<p><code><a href="https://dplyr.tidyverse.org/reference/filter.html">filter()</a></code> allows you to select a subset of rows in a data frame. Like all single verbs, the first argument is the tibble (or data frame). The second and subsequent arguments refer to variables within that data frame, selecting rows where the expression is <code>TRUE</code>.</p>
    
<p>For example, we can select all character with light skin color and brown eyes with:</p>

In [4]:
starwars |> filter(skin_color == "light", eye_color == "brown")

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Leia Organa,150,49,brown,light,brown,19,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,
Biggs Darklighter,183,84,black,light,brown,24,male,masculine,Tatooine,Human,A New Hope,,X-wing
Cordé,157,NA,brown,light,brown,NA,female,feminine,Naboo,Human,Attack of the Clones,,
Dormé,165,NA,brown,light,brown,NA,female,feminine,Naboo,Human,Attack of the Clones,,
Raymus Antilles,188,79,brown,light,brown,NA,male,masculine,Alderaan,Human,"Revenge of the Sith, A New Hope",,
Poe Dameron,NA,NA,brown,light,brown,NA,male,masculine,NA,Human,The Force Awakens,,T-70 X-wing fighter
Padmé Amidala,165,45,brown,light,brown,46,female,feminine,Naboo,Human,"Attack of the Clones, The Phantom Menace , Revenge of the Sith",,"H-type Nubian yacht, Naboo star skiff , Naboo fighter"


<p>This is roughly equivalent to this base R code:</p>

<pre><code>starwars[starwars$skin_color == "light" & starwars$eye_color == "brown"]</code></pre>

## <h3 id="arrange-rows-with-arrange">Arrange rows with <code>arrange()</code></h3>

<p><code><a href="https://dplyr.tidyverse.org/reference/arrange.html">arrange()</a></code> works similarly to <code><a href="https://dplyr.tidyverse.org/reference/filter.html">filter()</a></code> except that instead of filtering or selecting rows, it reorders them. It takes a data frame, and a set of column names (or more complicated expressions) to order by. If you provide more than one column name, each additional column will be used to break ties in the values of preceding columns:</p>

In [5]:
starwars |> arrange(height, mass)

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Yoda,66,17.0,white,green,brown,896,male,masculine,NA,Yoda's species,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi",,
Ratts Tyerell,79,15.0,none,"grey, blue",unknown,NA,male,masculine,Aleen Minor,Aleena,The Phantom Menace,,
Wicket Systri Warrick,88,20.0,brown,brown,brown,8,male,masculine,Endor,Ewok,Return of the Jedi,,
Dud Bolt,94,45.0,none,"blue, grey",yellow,NA,male,masculine,Vulpter,Vulptereen,The Phantom Menace,,
R2-D2,96,32.0,NA,"white, blue",red,33,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,
R4-P17,96,NA,none,"silver, red","red, blue",NA,none,feminine,NA,Droid,"Attack of the Clones, Revenge of the Sith",,
R5-D4,97,32.0,NA,"white, red",red,NA,none,masculine,Tatooine,Droid,A New Hope,,
Sebulba,112,40.0,none,"grey, red",orange,NA,male,masculine,Malastare,Dug,The Phantom Menace,,
Gasgano,122,NA,none,"white, blue",black,NA,male,masculine,Troiken,Xexto,The Phantom Menace,,


<p>Use <code><a href="https://dplyr.tidyverse.org/reference/desc.html">desc()</a></code> to order a column in descending order:</p>

In [6]:
starwars |> arrange(desc(height))

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Yarael Poof,264,NA,none,white,yellow,NA,male,masculine,Quermia,Quermian,The Phantom Menace,,
Tarfful,234,136,brown,brown,blue,NA,male,masculine,Kashyyyk,Wookiee,Revenge of the Sith,,
Lama Su,229,88,none,grey,black,NA,male,masculine,Kamino,Kaminoan,Attack of the Clones,,
Chewbacca,228,112,brown,unknown,blue,200.0,male,masculine,Kashyyyk,Wookiee,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",AT-ST,"Millennium Falcon, Imperial shuttle"
Roos Tarpals,224,82,none,grey,orange,NA,male,masculine,Naboo,Gungan,The Phantom Menace,,
Grievous,216,159,none,"brown, white","green, yellow",NA,male,masculine,Kalee,Kaleesh,Revenge of the Sith,Tsmeu-6 personal wheel bike,Belbullab-22 starfighter
Taun We,213,NA,none,grey,black,NA,female,feminine,Kamino,Kaminoan,Attack of the Clones,,
Rugor Nass,206,NA,none,green,orange,NA,male,masculine,Naboo,Gungan,The Phantom Menace,,
Tion Medon,206,80,none,grey,black,NA,male,masculine,Utapau,Pau'an,Revenge of the Sith,,


## <h3 id="choose-rows-using-their-position-with-slice">Choose rows using their position with <code>slice()</code></h3>

<p><code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice()</a></code> lets you index rows by their (integer) locations. It allows you to select, remove, and duplicate rows.</p>

<p>We can get characters from row numbers 5 through 10.</p>

In [7]:
starwars |> slice(5:10)

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Leia Organa,150,49,brown,light,brown,19,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,
Owen Lars,178,120,"brown, grey",light,blue,52,male,masculine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
Beru Whitesun lars,165,75,brown,light,blue,47,female,feminine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
R5-D4,97,32,NA,"white, red",red,NA,none,masculine,Tatooine,Droid,A New Hope,,
Biggs Darklighter,183,84,black,light,brown,24,male,masculine,Tatooine,Human,A New Hope,,X-wing
Obi-Wan Kenobi,182,77,"auburn, white",fair,blue-gray,57,male,masculine,Stewjon,Human,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",Tribubble bongo,"Jedi starfighter , Trade Federation cruiser, Naboo star skiff , Jedi Interceptor , Belbullab-22 starfighter"


<p>It is accompanied by a number of helpers for common use cases:</p>

<ul>
<li><code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice_head()</a></code> and <code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice_tail()</a></code> select the first or last rows.</li>
</ul>

In [8]:
starwars |> slice_head(n = 3)

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Luke Skywalker,172,77,blond,fair,blue,19,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens","Snowspeeder , Imperial Speeder Bike","X-wing , Imperial shuttle"
C-3PO,167,75,NA,gold,yellow,112,none,masculine,Tatooine,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",,
R2-D2,96,32,NA,"white, blue",red,33,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,


<ul>
<li><code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice_sample()</a></code> randomly selects rows. Use the option prop to choose a certain proportion of the cases.</li>
</ul>

In [9]:
starwars |> slice_sample(n = 5)

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Rey,NA,NA,brown,light,hazel,NA,female,feminine,NA,Human,The Force Awakens,,
Chewbacca,228,112,brown,unknown,blue,200,male,masculine,Kashyyyk,Wookiee,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",AT-ST,"Millennium Falcon, Imperial shuttle"
Arvel Crynyd,NA,NA,brown,fair,brown,NA,male,masculine,NA,Human,Return of the Jedi,,A-wing
Ayla Secura,178,55,none,blue,hazel,48,female,feminine,Ryloth,Twi'lek,"Attack of the Clones, The Phantom Menace , Revenge of the Sith",,
Raymus Antilles,188,79,brown,light,brown,NA,male,masculine,Alderaan,Human,"Revenge of the Sith, A New Hope",,


<p>Use <code>replace = TRUE</code> to perform a bootstrap sample. If needed, you can weight the sample with the <code>weight</code> argument.</p>

<ul>
<li><code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice_min()</a></code> and <code><a href="https://dplyr.tidyverse.org/reference/slice.html">slice_max()</a></code> select rows with highest or lowest values of a variable. Note that we first must choose only the values which are not NA.</li>
</ul>

In [10]:
starwars |>
filter(!is.na(height)) |>
  slice_max(height, n = 3)

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Yarael Poof,264,NA,none,white,yellow,NA,male,masculine,Quermia,Quermian,The Phantom Menace,,
Tarfful,234,136,brown,brown,blue,NA,male,masculine,Kashyyyk,Wookiee,Revenge of the Sith,,
Lama Su,229,88,none,grey,black,NA,male,masculine,Kamino,Kaminoan,Attack of the Clones,,


## <h3 id="select-columns-with-select">Select columns with <code>select()</code></h3>

<p>Often you work with large datasets with many columns but only a few are actually of interest to you. <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> allows you to rapidly zoom in on a useful subset using operations that usually only work on numeric variable positions:</p>

In [11]:
# Select columns by name
starwars |> select(hair_color, skin_color, eye_color)

hair_color,skin_color,eye_color
<chr>,<chr>,<chr>
blond,fair,blue
NA,gold,yellow
NA,"white, blue",red
none,white,yellow
brown,light,brown
"brown, grey",light,blue
brown,light,blue
NA,"white, red",red
black,light,brown


In [12]:
# Select all columns between hair_color and eye_color (inclusive)
starwars |> select(hair_color:eye_color)

hair_color,skin_color,eye_color
<chr>,<chr>,<chr>
blond,fair,blue
NA,gold,yellow
NA,"white, blue",red
none,white,yellow
brown,light,brown
"brown, grey",light,blue
brown,light,blue
NA,"white, red",red
black,light,brown


In [13]:
# Select all columns except those from hair_color to eye_color (inclusive)
starwars %>% select(!(hair_color:eye_color))

name,height,mass,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<chr>,<int>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Luke Skywalker,172,77.0,19.0,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens","Snowspeeder , Imperial Speeder Bike","X-wing , Imperial shuttle"
C-3PO,167,75.0,112.0,none,masculine,Tatooine,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",,
R2-D2,96,32.0,33.0,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,
Darth Vader,202,136.0,41.9,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope",,TIE Advanced x1
Leia Organa,150,49.0,19.0,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,
Owen Lars,178,120.0,52.0,male,masculine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
Beru Whitesun lars,165,75.0,47.0,female,feminine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
R5-D4,97,32.0,NA,none,masculine,Tatooine,Droid,A New Hope,,
Biggs Darklighter,183,84.0,24.0,male,masculine,Tatooine,Human,A New Hope,,X-wing


In [14]:
# Select all columns ending with color
starwars |> select(ends_with("color"))

hair_color,skin_color,eye_color
<chr>,<chr>,<chr>
blond,fair,blue
NA,gold,yellow
NA,"white, blue",red
none,white,yellow
brown,light,brown
"brown, grey",light,blue
brown,light,blue
NA,"white, red",red
black,light,brown


<p>There are a number of helper functions you can use within <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code>, like <code><a href="https://tidyselect.r-lib.org/reference/starts_with.html">starts_with()</a></code>, <code><a href="https://tidyselect.r-lib.org/reference/starts_with.html">ends_with()</a></code>, <code><a href="https://tidyselect.r-lib.org/reference/starts_with.html">matches()</a></code> and <code><a href="https://tidyselect.r-lib.org/reference/starts_with.html">contains()</a></code>. These let you quickly match larger blocks of variables that meet some criterion. See <code><a href="https://dplyr.tidyverse.org/reference/select.html">?select</a></code> for more details.</p>

<p>You can rename variables with <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> by using named arguments:</p>

In [15]:
starwars |> select(home_world = homeworld)

home_world
<chr>
Tatooine
Tatooine
Naboo
Tatooine
Alderaan
Tatooine
Tatooine
Tatooine
Tatooine


<p>But because <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> drops all the variables not explicitly mentioned, it’s not that useful. Instead, use <code><a href="https://dplyr.tidyverse.org/reference/rename.html">rename()</a></code>:</p>

In [16]:
starwars |> rename(home_world = homeworld)

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,home_world,species,films,vehicles,starships
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Luke Skywalker,172,77.0,blond,fair,blue,19.0,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens","Snowspeeder , Imperial Speeder Bike","X-wing , Imperial shuttle"
C-3PO,167,75.0,NA,gold,yellow,112.0,none,masculine,Tatooine,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",,
R2-D2,96,32.0,NA,"white, blue",red,33.0,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,
Darth Vader,202,136.0,none,white,yellow,41.9,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope",,TIE Advanced x1
Leia Organa,150,49.0,brown,light,brown,19.0,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,
Owen Lars,178,120.0,"brown, grey",light,blue,52.0,male,masculine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
Beru Whitesun lars,165,75.0,brown,light,blue,47.0,female,feminine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
R5-D4,97,32.0,NA,"white, red",red,NA,none,masculine,Tatooine,Droid,A New Hope,,
Biggs Darklighter,183,84.0,black,light,brown,24.0,male,masculine,Tatooine,Human,A New Hope,,X-wing


## <h3 id="add-new-columns-with-mutate">Add new columns with <code>mutate()</code></h3>

<p>Besides selecting sets of existing columns, it’s often useful to add new columns that are functions of existing columns. This is the job of
<code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code>:</p>

In [17]:
starwars |> mutate(height_m = height / 100)

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships,height_m
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>,<dbl>
Luke Skywalker,172,77.0,blond,fair,blue,19.0,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens","Snowspeeder , Imperial Speeder Bike","X-wing , Imperial shuttle",1.72
C-3PO,167,75.0,NA,gold,yellow,112.0,none,masculine,Tatooine,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",,,1.67
R2-D2,96,32.0,NA,"white, blue",red,33.0,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,,0.96
Darth Vader,202,136.0,none,white,yellow,41.9,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope",,TIE Advanced x1,2.02
Leia Organa,150,49.0,brown,light,brown,19.0,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,,1.50
Owen Lars,178,120.0,"brown, grey",light,blue,52.0,male,masculine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,,1.78
Beru Whitesun lars,165,75.0,brown,light,blue,47.0,female,feminine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,,1.65
R5-D4,97,32.0,NA,"white, red",red,NA,none,masculine,Tatooine,Droid,A New Hope,,,0.97
Biggs Darklighter,183,84.0,black,light,brown,24.0,male,masculine,Tatooine,Human,A New Hope,,X-wing,1.83


<p>We can’t see the height in meters we just calculated, but we can fix that using a select command.</p>

In [18]:
starwars |>
  mutate(height_m = height / 100) |>
  select(height_m, height, everything())

height_m,height,name,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<dbl>,<int>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
1.72,172,Luke Skywalker,77.0,blond,fair,blue,19.0,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens","Snowspeeder , Imperial Speeder Bike","X-wing , Imperial shuttle"
1.67,167,C-3PO,75.0,NA,gold,yellow,112.0,none,masculine,Tatooine,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",,
0.96,96,R2-D2,32.0,NA,"white, blue",red,33.0,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,
2.02,202,Darth Vader,136.0,none,white,yellow,41.9,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope",,TIE Advanced x1
1.50,150,Leia Organa,49.0,brown,light,brown,19.0,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,
1.78,178,Owen Lars,120.0,"brown, grey",light,blue,52.0,male,masculine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
1.65,165,Beru Whitesun lars,75.0,brown,light,blue,47.0,female,feminine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
0.97,97,R5-D4,32.0,NA,"white, red",red,NA,none,masculine,Tatooine,Droid,A New Hope,,
1.83,183,Biggs Darklighter,84.0,black,light,brown,24.0,male,masculine,Tatooine,Human,A New Hope,,X-wing


<p><code><a href="https://dplyr.tidyverse.org/reference/mutate.html">dplyr::mutate()</a></code> is similar to the base <code><a href="https://rdrr.io/r/base/transform.html">transform()</a></code>, but allows you to refer to columns that you’ve just created:</p>

In [19]:
starwars |>
  mutate(
    height_m = height / 100,
    BMI = mass / (height_m^2)
  ) |>
  select(everything())

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships,height_m,BMI
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>,<dbl>,<dbl>
Luke Skywalker,172,77.0,blond,fair,blue,19.0,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens","Snowspeeder , Imperial Speeder Bike","X-wing , Imperial shuttle",1.72,26.02758
C-3PO,167,75.0,NA,gold,yellow,112.0,none,masculine,Tatooine,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",,,1.67,26.89232
R2-D2,96,32.0,NA,"white, blue",red,33.0,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,,0.96,34.72222
Darth Vader,202,136.0,none,white,yellow,41.9,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope",,TIE Advanced x1,2.02,33.33007
Leia Organa,150,49.0,brown,light,brown,19.0,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,,1.50,21.77778
Owen Lars,178,120.0,"brown, grey",light,blue,52.0,male,masculine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,,1.78,37.87401
Beru Whitesun lars,165,75.0,brown,light,blue,47.0,female,feminine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,,1.65,27.54821
R5-D4,97,32.0,NA,"white, red",red,NA,none,masculine,Tatooine,Droid,A New Hope,,,0.97,34.00999
Biggs Darklighter,183,84.0,black,light,brown,24.0,male,masculine,Tatooine,Human,A New Hope,,X-wing,1.83,25.08286


<p>If you only want to keep the new variables, use <code>.keep = "none"</code>:</p>

In [20]:
starwars |>
  mutate(
    height_m = height / 100,
    BMI = mass / (height_m^2),
    .keep = "none"
  )

height_m,BMI
<dbl>,<dbl>
1.72,26.02758
1.67,26.89232
0.96,34.72222
2.02,33.33007
1.50,21.77778
1.78,37.87401
1.65,27.54821
0.97,34.00999
1.83,25.08286


## <h3 id="change-column-order-with-relocate">Change column order with <code>relocate()</code></h3>

<p>Use a similar syntax as <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> to move blocks of columns at once</p>

In [21]:
starwars |> relocate(sex:homeworld, .before = height)

name,sex,gender,homeworld,height,mass,hair_color,skin_color,eye_color,birth_year,species,films,vehicles,starships
<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<list>,<list>,<list>
Luke Skywalker,male,masculine,Tatooine,172,77.0,blond,fair,blue,19.0,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens","Snowspeeder , Imperial Speeder Bike","X-wing , Imperial shuttle"
C-3PO,none,masculine,Tatooine,167,75.0,NA,gold,yellow,112.0,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",,
R2-D2,none,masculine,Naboo,96,32.0,NA,"white, blue",red,33.0,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,
Darth Vader,male,masculine,Tatooine,202,136.0,none,white,yellow,41.9,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope",,TIE Advanced x1
Leia Organa,female,feminine,Alderaan,150,49.0,brown,light,brown,19.0,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,
Owen Lars,male,masculine,Tatooine,178,120.0,"brown, grey",light,blue,52.0,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
Beru Whitesun lars,female,feminine,Tatooine,165,75.0,brown,light,blue,47.0,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
R5-D4,none,masculine,Tatooine,97,32.0,NA,"white, red",red,NA,Droid,A New Hope,,
Biggs Darklighter,male,masculine,Tatooine,183,84.0,black,light,brown,24.0,Human,A New Hope,,X-wing


## <h3 id="summarise-values-with-summarise">Summarise values with <code>summarise()</code></h3>

<p>The last verb is <code><a href="https://dplyr.tidyverse.org/reference/summarise.html">summarise()</a></code>. It collapses a data frame to a single row.</p>

In [22]:
starwars |> summarise(height = mean(height, na.rm = TRUE))

height
<dbl>
174.358


<p>It’s not that useful until we learn the <code><a href="https://dplyr.tidyverse.org/reference/group_by.html">group_by()</a></code> verb below.</p>

## <h3 id="commonalities">Commonalities</h3>

<p>You may have noticed that the syntax and function of all these verbs are very similar:</p>

<ul>
<li><p>The first argument is a data frame.</p></li>
<li><p>The subsequent arguments describe what to do with the data frame. You can refer to columns in the data frame directly without using <code>$</code>.</p></li>
<li><p>The result is a new data frame</p></li>
</ul>

<p>Together these properties make it easy to chain together multiple simple steps to achieve a complex result.</p>

<p>These five functions provide the basis of a language of data manipulation. At the most basic level, you can only alter a tidy data frame in five useful ways: you can reorder the rows (<code><a href="https://dplyr.tidyverse.org/reference/arrange.html">arrange()</a></code>), pick observations and variables of interest (<code><a href="https://dplyr.tidyverse.org/reference/filter.html">filter()</a></code> and <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code>), add new variables that are functions of existing variables (<code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code>), or collapse many values to a summary (<code><a href="https://dplyr.tidyverse.org/reference/summarise.html">summarise()</a></code>).</p>

## <h2 id="combining-functions-with">Combining functions with <code>%&gt;%</code></h2>

<p>The dplyr API is functional in the sense that function calls don’t have side-effects. You must always save their results. This doesn’t lead
to particularly elegant code, especially if you want to do many operations at once. You either have to do it step-by-step:</p>

<pre><code>a1 <- <a href="https://dplyr.tidyverse.org/reference/group_by.html">group_by</a>(starwars, species, sex)
a2 <- <a href="https://dplyr.tidyverse.org/reference/select.html">select</a>(a1, height, mass)
a3 <- <a href="https://dplyr.tidyverse.org/reference/summarise.html">summarise</a>(a2,
  height = <a href="https://rdrr.io/r/base/mean.html">mean</a>(height, na.rm = TRUE),
  mass = <a href="https://rdrr.io/r/base/mean.html">mean</a>(mass, na.rm = TRUE)
)</code></pre>

<p>Or if you don’t want to name the intermediate results, you need to wrap the function calls inside each other:</p>

In [23]:
summarise(
  select(
    group_by(starwars, species, sex),
    height, mass
  ),
  height = mean(height, na.rm = TRUE),
  mass = mean(mass, na.rm = TRUE)
)

Adding missing grouping variables: `species`, `sex`
`summarise()` has grouped output by 'species'. You can override using the `.groups` argument.


species,sex,height,mass
<chr>,<chr>,<dbl>,<dbl>
Aleena,male,79.0000,15.00000
Besalisk,male,198.0000,102.00000
Cerean,male,198.0000,82.00000
Chagrian,male,196.0000,NaN
Clawdite,female,168.0000,55.00000
Droid,none,131.2000,69.75000
Dug,male,112.0000,40.00000
Ewok,male,88.0000,20.00000
Geonosian,male,183.0000,80.00000


<p>This is difficult to read because the order of the operations is from inside to out. Thus, the arguments are a long way away from the
function. To get around this problem, dplyr provides the <code>%&gt;%</code> operator from magrittr. <code>x %&gt;% f(y)</code> turns into <code>f(x, y)</code> so you can use it to rewrite multiple operations that you can read left-to-right, top-to-bottom (reading the pipe operator as “then”):</p>

<pre><code>starwars <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
  <a href="https://dplyr.tidyverse.org/reference/group_by.html">group_by</a>(<species, sex) <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
  <a href="https://dplyr.tidyverse.org/reference/select.html">select</a>(height, mass) <a href="https://magrittr.tidyverse.org/reference/pipe.html">%&gt;%</a>
  <a href="https://dplyr.tidyverse.org/reference/summarise.html">summarise</a>(
    height = <a href="https://rdrr.io/r/base/mean.html">mean</a>(height, na.rm = TRUE),
    mass = <a href="https://rdrr.io/r/base/mean.html">mean</a>(mass, na.rm = TRUE)
  )</code></pre>

# <h2 id="patterns-of-operations">Patterns of operations</h2>

<p>The dplyr verbs can be classified by the type of operations they accomplish (we sometimes speak of their <strong>semantics</strong>, i.e., their meaning). It’s helpful to have a good grasp of the difference between select and mutate operations.</p>

## <h3 id="selecting-operations">Selecting operations</h3>

<p>One of the appealing features of dplyr is that you can refer to columns from the tibble as if they were regular variables. However, the syntactic uniformity of referring to bare column names hides semantical differences across the verbs. A column symbol supplied to <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> does not have the same meaning as the same symbol supplied to <code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code>.</p>

<p>Selecting operations expect column names and positions. Hence, when you call <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> with bare variable names, they actually represent their own positions in the tibble. The following calls are completely equivalent from dplyr’s point of view:</p>

In [24]:
# `name` represents the integer 1
select(starwars, name)

name
<chr>
Luke Skywalker
C-3PO
R2-D2
Darth Vader
Leia Organa
Owen Lars
Beru Whitesun lars
R5-D4
Biggs Darklighter


In [25]:
select(starwars, 1)

name
<chr>
Luke Skywalker
C-3PO
R2-D2
Darth Vader
Leia Organa
Owen Lars
Beru Whitesun lars
R5-D4
Biggs Darklighter


<p>By the same token, this means that you cannot refer to variables from the surrounding context if they have the same name as one of the columns. In the following example, <code>height</code> still represents 2, not 5:</p>

In [26]:
height <- 5
select(starwars, height)

height
<int>
172
167
96
202
150
178
165
97
183


<p>One useful subtlety is that this only applies to bare names and to selecting calls like <code>c(height, mass)</code> or <code>height:mass</code>. In all other cases, the columns of the data frame are not put in scope. This allows you to refer to contextual variables in selection helpers:</p>

In [27]:
name <- "color"
select(starwars, ends_with(name))

hair_color,skin_color,eye_color
<chr>,<chr>,<chr>
blond,fair,blue
NA,gold,yellow
NA,"white, blue",red
none,white,yellow
brown,light,brown
"brown, grey",light,blue
brown,light,blue
NA,"white, red",red
black,light,brown


<p>These semantics are usually intuitive. But note the subtle difference:</p>

In [28]:
name <- 5
select(starwars, name, identity(name))

name,skin_color
<chr>,<chr>
Luke Skywalker,fair
C-3PO,gold
R2-D2,"white, blue"
Darth Vader,white
Leia Organa,light
Owen Lars,light
Beru Whitesun lars,light
R5-D4,"white, red"
Biggs Darklighter,light


<p>In the first argument, <code>name</code> represents its own position <code>1</code>. In the second argument, <code>name</code> is evaluated in the surrounding context and represents the fifth column.</p>

<p>For a long time, <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> used to only understand column positions. Counting from dplyr 0.6, it now understands column names as well. This makes it a bit easier to program with <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code>:</p>

In [29]:
vars <- c("name", "height")
select(starwars, all_of(vars), "mass")

name,height,mass
<chr>,<int>,<dbl>
Luke Skywalker,172,77.0
C-3PO,167,75.0
R2-D2,96,32.0
Darth Vader,202,136.0
Leia Organa,150,49.0
Owen Lars,178,120.0
Beru Whitesun lars,165,75.0
R5-D4,97,32.0
Biggs Darklighter,183,84.0


## <h3 id="mutating-operations">Mutating operations</h3>

<p>Mutate semantics are quite different from selection semantics.
Whereas <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code> expects column names or positions, <code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code> expects <em>column vectors</em>. We will set up a smaller tibble to use for our examples.</p>

In [30]:
df <- starwars |> select(name, height, mass)

<p>When we use <code><a href="https://dplyr.tidyverse.org/reference/select.html">select()</a></code>, the bare column names stand for their own positions in the tibble. For <code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code> on the other hand, column symbols represent the actual column vectors stored in the tibble. Consider what happens if we give a string or a number to <code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code>:</p>

In [31]:
mutate(df, "height", 2)

name,height,mass,"""height""",2
<chr>,<int>,<dbl>,<chr>,<dbl>
Luke Skywalker,172,77.0,height,2
C-3PO,167,75.0,height,2
R2-D2,96,32.0,height,2
Darth Vader,202,136.0,height,2
Leia Organa,150,49.0,height,2
Owen Lars,178,120.0,height,2
Beru Whitesun lars,165,75.0,height,2
R5-D4,97,32.0,height,2
Biggs Darklighter,183,84.0,height,2


<p><code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code> gets length-1 vectors that it interprets as new columns in the data frame. These vectors are recycled so they match the number of rows. That’s why it doesn’t make sense to supply expressions like <code>"height" + 10</code> to <code><a href="https://dplyr.tidyverse.org/reference/mutate.html">mutate()</a></code>. This amounts to adding 10 to a string! The correct expression is:</p>

In [32]:
mutate(df, height + 10)

name,height,mass,height + 10
<chr>,<int>,<dbl>,<dbl>
Luke Skywalker,172,77.0,182
C-3PO,167,75.0,177
R2-D2,96,32.0,106
Darth Vader,202,136.0,212
Leia Organa,150,49.0,160
Owen Lars,178,120.0,188
Beru Whitesun lars,165,75.0,175
R5-D4,97,32.0,107
Biggs Darklighter,183,84.0,193


<p>In the same way, you can unquote values from the context if these values represent a valid column. They must be either length 1 (they then get recycled) or have the same length as the number of rows. In the following example we create a new vector that we add to the data frame:</p>

In [33]:
var <- seq(1, nrow(df))
mutate(df, new = var)

name,height,mass,new
<chr>,<int>,<dbl>,<int>
Luke Skywalker,172,77.0,1
C-3PO,167,75.0,2
R2-D2,96,32.0,3
Darth Vader,202,136.0,4
Leia Organa,150,49.0,5
Owen Lars,178,120.0,6
Beru Whitesun lars,165,75.0,7
R5-D4,97,32.0,8
Biggs Darklighter,183,84.0,9


<p>A case in point is <code><a href="https://dplyr.tidyverse.org/reference/group_by.html">group_by()</a></code>. While you might think it has select semantics, it actually has mutate semantics. This is quite handy as it allows to group by a modified column:</p>

In [36]:
group_by(starwars, sex)

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Luke Skywalker,172,77.0,blond,fair,blue,19.0,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens","Snowspeeder , Imperial Speeder Bike","X-wing , Imperial shuttle"
C-3PO,167,75.0,NA,gold,yellow,112.0,none,masculine,Tatooine,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",,
R2-D2,96,32.0,NA,"white, blue",red,33.0,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,
Darth Vader,202,136.0,none,white,yellow,41.9,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope",,TIE Advanced x1
Leia Organa,150,49.0,brown,light,brown,19.0,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,
Owen Lars,178,120.0,"brown, grey",light,blue,52.0,male,masculine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
Beru Whitesun lars,165,75.0,brown,light,blue,47.0,female,feminine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
R5-D4,97,32.0,NA,"white, red",red,NA,none,masculine,Tatooine,Droid,A New Hope,,
Biggs Darklighter,183,84.0,black,light,brown,24.0,male,masculine,Tatooine,Human,A New Hope,,X-wing


In [37]:
group_by(starwars, sex = as.factor(sex))

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<fct>,<chr>,<chr>,<chr>,<list>,<list>,<list>
Luke Skywalker,172,77.0,blond,fair,blue,19.0,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens","Snowspeeder , Imperial Speeder Bike","X-wing , Imperial shuttle"
C-3PO,167,75.0,NA,gold,yellow,112.0,none,masculine,Tatooine,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",,
R2-D2,96,32.0,NA,"white, blue",red,33.0,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,
Darth Vader,202,136.0,none,white,yellow,41.9,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope",,TIE Advanced x1
Leia Organa,150,49.0,brown,light,brown,19.0,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,
Owen Lars,178,120.0,"brown, grey",light,blue,52.0,male,masculine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
Beru Whitesun lars,165,75.0,brown,light,blue,47.0,female,feminine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,
R5-D4,97,32.0,NA,"white, red",red,NA,none,masculine,Tatooine,Droid,A New Hope,,
Biggs Darklighter,183,84.0,black,light,brown,24.0,male,masculine,Tatooine,Human,A New Hope,,X-wing


In [38]:
group_by(starwars, height_binned = cut(height, 3))

name,height,mass,hair_color,skin_color,eye_color,birth_year,sex,gender,homeworld,species,films,vehicles,starships,height_binned
<chr>,<int>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<list>,<list>,<list>,<fct>
Luke Skywalker,172,77.0,blond,fair,blue,19.0,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens","Snowspeeder , Imperial Speeder Bike","X-wing , Imperial shuttle","(132,198]"
C-3PO,167,75.0,NA,gold,yellow,112.0,none,masculine,Tatooine,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope",,,"(132,198]"
R2-D2,96,32.0,NA,"white, blue",red,33.0,none,masculine,Naboo,Droid,"The Empire Strikes Back, Attack of the Clones , The Phantom Menace , Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",,,"(65.8,132]"
Darth Vader,202,136.0,none,white,yellow,41.9,male,masculine,Tatooine,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope",,TIE Advanced x1,"(198,264]"
Leia Organa,150,49.0,brown,light,brown,19.0,female,feminine,Alderaan,Human,"The Empire Strikes Back, Revenge of the Sith , Return of the Jedi , A New Hope , The Force Awakens",Imperial Speeder Bike,,"(132,198]"
Owen Lars,178,120.0,"brown, grey",light,blue,52.0,male,masculine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,,"(132,198]"
Beru Whitesun lars,165,75.0,brown,light,blue,47.0,female,feminine,Tatooine,Human,"Attack of the Clones, Revenge of the Sith , A New Hope",,,"(132,198]"
R5-D4,97,32.0,NA,"white, red",red,NA,none,masculine,Tatooine,Droid,A New Hope,,,"(65.8,132]"
Biggs Darklighter,183,84.0,black,light,brown,24.0,male,masculine,Tatooine,Human,A New Hope,,X-wing,"(132,198]"


<p>This is why you can’t supply a column name to <code><a href="https://dplyr.tidyverse.org/reference/group_by.html">group_by()</a></code>. This amounts to creating a new column containing the string recycled to the number of rows:</p>

In [39]:
group_by(df, "month")

name,height,mass,"""month"""
<chr>,<int>,<dbl>,<chr>
Luke Skywalker,172,77.0,month
C-3PO,167,75.0,month
R2-D2,96,32.0,month
Darth Vader,202,136.0,month
Leia Organa,150,49.0,month
Owen Lars,178,120.0,month
Beru Whitesun lars,165,75.0,month
R5-D4,97,32.0,month
Biggs Darklighter,183,84.0,month
